<a href="https://colab.research.google.com/github/sahanasb/Intelligent-E-Commerce-Search-with-Retrieval-Augmented-Generation/blob/main/BM25%2BSBERT%2BHybridSearch%2BRedis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas rank_bm25 sentence-transformers scikit-learn

In [ ]:
!pip install langchain langchain-community langchain-groq sentence-transformers chromadb --break-system-packages

In [6]:
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import torch
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np
import shutil
import os

# loading data
df = pd.read_csv('/content/sample_data/clean_data.csv')
df.head()

,name,main_category,image,ratings,no_of_ratings,actual_price,source_file,actual_price_usd,ranking_score,product_id,search_text
0,Aston Andia Women Waist Cinchers Corset Adjust...,Womens Innerwear,https://m.media-amazon.com/images/I/51qLMr3Fea...,4.0,43,999,Womens Innerwear.csv,10.99,3.68,0,product: aston andia women waist cinchers cors...
1,Floret Women's Cotton Non Padded Wire Free T-S...,Womens Innerwear,https://m.media-amazon.com/images/I/61sBzjDk7W...,4.0,1,399,Womens Innerwear.csv,4.39,2.91,1,product: floret women's cotton non padded wire...
2,DOT WAVE Women's Cotton Pyjamas - Loungepants ...,Womens Innerwear,https://m.media-amazon.com/images/I/81qQK0+n4p...,4.0,153,1699,Womens Innerwear.csv,18.69,3.89,2,product: dot wave women's cotton pyjamas - lou...
3,Bali womens Comfort Revolution Wireless T-shir...,Womens Innerwear,https://m.media-amazon.com/images/I/81ReKuvZQr...,4.0,9328,1500,Womens Innerwear.csv,16.50,4.25,3,product: bali womens comfort revolution wirele...
4,TRYLO Women's Non-Wired Bra,Womens Innerwear,https://m.media-amazon.com/images/W/IMAGERENDE...,4.0,11,435,Womens Innerwear.csv,4.78,3.43,4,product: trylo women's non-wired bra. category...


In [7]:
# Vector embedding
def build_doc(row, index):
    return Document(
        page_content=str(row['search_text']),
        metadata={
            # add index for Hybrid Search
            "idx": index,
            "name": str(row['name']),
            "category": str(row['main_category']),
            "price_usd": float(row['actual_price_usd']),
            "ratings": float(row['ratings'])
        }
    )
docs = [build_doc(row, index) for index, row in df.iterrows()]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"})

/tmp/ipykernel_5329/1896174876.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
if os.path.exists("/content/sample_data/product_db"):
    shutil.rmtree("/content/sample_data/product_db")
    print("Deleted old product_db")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="/content/sample_data/product_db"
)

In [13]:
results = vectorstore.similarity_search("i am planning to go to beach")

for i, doc in enumerate(results[:3]):
    print(f"\n--- Document {i+1} ---")
    print("Name:", doc.metadata.get("name"))
    print("Category:", doc.metadata.get("category"))
    print("Price:", doc.metadata.get("price_usd"))
    print("Rating:", doc.metadata.get("ratings"))
    print("Content:", doc.page_content)


--- Document 1 ---
Name: Beach Toy Set for Kids 8 Pcs with Castle Shaped Bucket Shovels Water Sprinkler and Moulds Made in India Perfect Beach Toy ...
Category: Toys
Price: 3.58
Rating: 4.4
Content: product: beach toy set for kids 8 pcs with castle shaped bucket shovels water sprinkler and moulds made in india perfect beach toy .... category: toys. price: $3.58. rating: 4.4 out of 5. reviews: 13. beach toy set for kids 8 pcs with castle shaped bucket shovels water sprinkler and moulds made in india perfect beach toy ...

--- Document 2 ---
Name: BEREAL Macaron -White Cork Sandal Women
Category: Womens Sandals
Price: 14.29
Rating: 5.0
Content: product: bereal macaron -white cork sandal women. category: womens sandals. price: $14.29. rating: 5.0 out of 5. reviews: 1. bereal macaron -white cork sandal women

--- Document 3 ---
Name: Inc.5 womens 1191_peach Sandal
Category: Womens Sandals
Price: 27.39
Rating: 5.0
Content: product: inc.5 womens 1191_peach sandal. category: womens sandals. 

#BM25

In [14]:
# Tokenization
tokenized_corpus = [doc.page_content.lower().split() for doc in docs]

# Initialization
bm25 = BM25Okapi(tokenized_corpus)

# BM25 query
def bm25_search(query, top_k=5):
  # Preprocess the user's query
  tokenized_query = query.lower().split()

  # Calculate scores for all documents in the corpus
  bm25_scores = bm25.get_scores(tokenized_query)

  # Get indices of top_k results
  top_indices = np.argsort(bm25_scores)[::-1][:top_k]
  print(f"Query: {query}")

  for i in top_indices:
      print(f"Score: {bm25_scores[i]:.2f} | Name: {df.iloc[i]['name']}")
  return top_indices, bm25_scores
# Test BM25
indices, scores = bm25_search("i am planning to go to beach")

Query: i am planning to go to beach
Score: 22.47 | Name: Hangout Hub Couple Men's & Women's Cotton Printed Regular Fit T-Shirts (Pack of 2) - I Am Her King I Am His Queen
Score: 19.71 | Name: Jungle Magic Doodle Waterz - Reusable I Water Colouring Book - Vehicles I Self-Drying with Easy to Hold Water Pen I Educat...
Score: 15.84 | Name: SanDisk Extreme 64GB CompactFlash Memory Card UDMA 7 Speed Up to 120MB/s & Extreme Pro SD UHS I 128GB Card for 4K Video fo...
Score: 15.54 | Name: E2B Adaptor/E-Type to B-Type Mount/elinchrom to Bowens/for Studio Light/Quick Lock/All-Metal/Spring-Loaded/elinchrom to godox
Score: 14.19 | Name: PARTY PROPZ Bachelorette Combo 1 Bride to BE Eye Glass, 1 Bride to BE SASH & 1 Set of Bride to BE PHOTOBOOTH /Bachelorette...


#SBERT

In [15]:
#SBERT
def sbert_search(query, top_k=5):
  # convert a single string of text into a vector
  query_embedding = embeddings.embed_query(query)

  # finds the closest matches using metrics like cosine similarity
  results = vectorstore.similarity_search(query, k=top_k)
  return results

# Test SBERT
query = "i am planning to go to beach"
sbert_results = sbert_search(query)
for doc in sbert_results:
    print(f"- {doc.metadata['name']}")

- Beach Toy Set for Kids 8 Pcs with Castle Shaped Bucket Shovels Water Sprinkler and Moulds Made in India Perfect Beach Toy ...
- BEREAL Macaron -White Cork Sandal Women
- Inc.5 womens 1191_peach Sandal
- Skechers Womens Go Walk 5 Foamies - Pup Life Sandal
- Skechers womens Max Cushioning Sandal


#Hybrid Search

In [16]:
def hybrid_search(query, top_k=5):
  # get BM25 rank
  tokenized_query = query.lower().split()
  bm25_scores = bm25.get_scores(tokenized_query)
  bm25_top_indices = np.argsort(bm25_scores)[::-1]
  bm25_mapping = {idx: rank for rank, idx in enumerate(bm25_top_indices)}

  # get SBERT rank
  sbert_results = vectorstore.similarity_search(query, k=100)

  sbert_mapping = {}
  for rank, doc in enumerate(sbert_results):
    idx = doc.metadata.get('idx')
    sbert_mapping[idx] = rank

  # Reciprocal Rank Fusion (RRF) to combine two ranked results
  rrf_scores = {}
  k = 60
  all_candidate_indices = set(list(bm25_top_indices[:100]) + list(sbert_mapping.keys()))

  for idx in all_candidate_indices:
    rank_bm = bm25_mapping.get(idx, 1000)
    rank_sbert = sbert_mapping.get(idx, 1000)

    score = (1 / (k + rank_bm)) + (1 / (k + rank_sbert))
    rrf_scores[idx] = score
  # sort score
  final_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
  return [docs[i] for i in final_indices]

# Test hybrid search
querty = "i am planning to go to beach"
hybrid_results = hybrid_search(query)
for doc in hybrid_results:
    print(f"- {doc.metadata['name']}")

- Beach Toy Set for Kids 8 Pcs with Castle Shaped Bucket Shovels Water Sprinkler and Moulds Made in India Perfect Beach Toy ...
- Skechers Womens Go Walk 5 Foamies - Pup Life Sandal
- BEREAL Macaron -White Cork Sandal Women
- WELCOME Women's Tan Sandal-6 Kids UK (LC 11)
- Loblique Comfort Sandal for Women's & Girl's BLUEDAMRU_3


# Redis

In [17]:
pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 10.8 MB/s eta 0:00:00


In [18]:
!apt-get install redis-server > /dev/null
!redis-server --daemonize yes

In [19]:
import redis
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
try:
  if r.ping():
    print("connect redis successfully")
except Exception as e:
    print("not connect redis")

connect redis successfully


In [20]:
import json
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
def final_search_pipeline(query, top_k=5):
  start_time = time.time()

  # check redis cache
  cache_key = f"search:{query.lower().strip()}"
  cached_res = r.get(cache_key)

  if cached_res:
    latency = (time.time() - start_time) * 1000
    print(f"CATCH HIT running: {latency:.2f}ms")
    return json.loads(cached_res)

  # query not in cache
  initial_candidates = hybrid_search(query, top_k=50)
  final_results = initial_candidates[:top_k]

  # TODO rerank_result then switch initial_candidates[:top_k] to rerank_results(query, initial_candidates, top_n=top_k)

  #final_results = rerank_results(query, initial_candidates, top_n=top_k)
  output = []
  for doc in final_results:
    output.append({
        "name": doc.metadata.get("name"),
        "price": doc.metadata.get("price_usd"),
        "category": doc.metadata.get("category"),
        "idx": doc.metadata.get("idx")
    })
  r.setex(cache_key, 3600, json.dumps(output))
  return output
results = final_search_pipeline("i am planning to go to beach")

**Evaluationt**

In [21]:
# ── Install reranker ──────────────────────────────────────────────
!pip install sentence-transformers --quiet  # already present, ensures CrossEncoder

In [22]:
# ── Build synthetic evaluation dataset ────────────────────────────
import pandas as pd
import numpy as np
import random

random.seed(42)

# Sample N products as eval queries – their own idx is the ground-truth relevant doc
N_EVAL = 100
eval_sample = df.sample(N_EVAL, random_state=42).reset_index(drop=True)

# Each record: {"query": search_text, "relevant_idx": original df index}
eval_set = [
    {"query": row["search_text"], "relevant_idx": row.name}   # row.name = df index
    for _, row in eval_sample.iterrows()
]

print(f"Eval queries: {len(eval_set)}")
print("Example:", eval_set[0])

Eval queries: 100
Example: {'query': 'product: coocaa 164 cm (65 inches) frameless series 4k ultra hd smart ips google led tv 65y72 (black). category: televisions. price: $1099.99. rating: 4.2 out of 5. reviews: 80. coocaa 164 cm (65 inches) frameless series 4k ultra hd smart ips google led tv 65y72 (black)', 'relevant_idx': 0}


In [23]:
# ── Recall@K helpers ──────────────────────────────────────────────
def recall_at_k(retrieved_indices: list, relevant_idx: int) -> int:
    """Returns 1 if relevant_idx is in the retrieved list, else 0."""
    return 1 if relevant_idx in retrieved_indices else 0


def evaluate_recall(retriever_fn, eval_set, K=10):
    hits = []
    for item in eval_set:
        query       = item["query"]
        relevant    = item["relevant_idx"]
        top_indices = retriever_fn(query, top_k=K)
        hits.append(recall_at_k(top_indices, relevant))
    return np.mean(hits)

In [24]:
# ── Wrap each retriever to return index lists ─────────────────────

def bm25_retriever(query, top_k=10):
    tokenized_query = query.lower().split()
    bm25_scores     = bm25.get_scores(tokenized_query)
    return list(np.argsort(bm25_scores)[::-1][:top_k])


def sbert_retriever(query, top_k=10):
    results = vectorstore.similarity_search(query, k=top_k)
    return [doc.metadata["idx"] for doc in results]


def hybrid_retriever(query, top_k=10):
    results = hybrid_search(query, top_k=top_k)   # existing function
    return [doc.metadata["idx"] for doc in results]

In [25]:
# ── Run Stage 1 ───────────────────────────────────────────────────
K = 10

r_bm25   = evaluate_recall(bm25_retriever,   eval_set, K=K)
r_sbert  = evaluate_recall(sbert_retriever,  eval_set, K=K)
r_hybrid = evaluate_recall(hybrid_retriever, eval_set, K=K)

stage1 = pd.DataFrame({
    "Retriever": ["BM25", "SBERT", "Hybrid (RRF)"],
    f"Recall@{K}": [r_bm25, r_sbert, r_hybrid]
})
print(stage1.to_string(index=False))

   Retriever  Recall@10
        BM25        0.0
       SBERT        0.0
Hybrid (RRF)        0.0


In [26]:
# ── CrossEncoder reranker ────────────────────────────────────────
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_results(query: str, candidate_docs: list, top_n: int = 5) -> list:
    """
    Reranks a list of LangChain Document objects using a CrossEncoder.
    Returns top_n re-ranked documents.
    """
    pairs   = [(query, doc.page_content) for doc in candidate_docs]
    scores  = cross_encoder.predict(pairs)
    ranked  = sorted(zip(scores, candidate_docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_n]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [27]:
# ── Full pipeline builders (retriever → reranker) ────────────────

def bm25_then_rerank(query, top_k=5, candidate_pool=50):
    tokenized = query.lower().split()
    scores    = bm25.get_scores(tokenized)
    top_idx   = list(np.argsort(scores)[::-1][:candidate_pool])
    candidates = [docs[i] for i in top_idx]
    reranked   = rerank_results(query, candidates, top_n=top_k)
    return [doc.metadata["idx"] for doc in reranked]


def sbert_then_rerank(query, top_k=5, candidate_pool=50):
    candidates = vectorstore.similarity_search(query, k=candidate_pool)
    reranked   = rerank_results(query, candidates, top_n=top_k)
    return [doc.metadata["idx"] for doc in reranked]


def hybrid_then_rerank(query, top_k=5, candidate_pool=50):
    candidates = hybrid_search(query, top_k=candidate_pool)
    reranked   = rerank_results(query, candidates, top_n=top_k)
    return [doc.metadata["idx"] for doc in reranked]

In [28]:
# ── Precision@K helper ───────────────────────────────────────────
def precision_at_k(retrieved_indices: list, relevant_idx: int, K: int) -> float:
    hits = sum(1 for idx in retrieved_indices[:K] if idx == relevant_idx)
    return hits / K


def evaluate_precision(pipeline_fn, eval_set, K=5):
    scores = []
    for item in eval_set:
        query    = item["query"]
        relevant = item["relevant_idx"]
        top_idx  = pipeline_fn(query, top_k=K)
        scores.append(precision_at_k(top_idx, relevant, K))
    return np.mean(scores)

In [ ]:
# ── Run Stage 2 ──────────────────────────────────────────────────
K_RERANK = 5

p_bm25_rr   = evaluate_precision(bm25_then_rerank,   eval_set, K=K_RERANK)
p_sbert_rr  = evaluate_precision(sbert_then_rerank,  eval_set, K=K_RERANK)
p_hybrid_rr = evaluate_precision(hybrid_then_rerank, eval_set, K=K_RERANK)

stage2 = pd.DataFrame({
    "Pipeline":       ["BM25 + Rerank", "SBERT + Rerank", "Hybrid + Rerank"],
    f"Precision@{K_RERANK}": [p_bm25_rr, p_sbert_rr, p_hybrid_rr]
})
print(stage2.to_string(index=False))

In [ ]:
# ── Combined results table + bar chart ──────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Stage 1 chart
axes[0].bar(stage1["Retriever"], stage1[f"Recall@{K}"],
            color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title(f"Stage 1 – Recall@{K} (Base Retrievers)")
axes[0].set_ylabel("Recall@K")
axes[0].set_ylim(0, 1.0)
for i, v in enumerate(stage1[f"Recall@{K}"]):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")

# Stage 2 chart
axes[1].bar(stage2["Pipeline"], stage2[f"Precision@{K_RERANK}"],
            color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title(f"Stage 2 – Precision@{K_RERANK} (After Reranking)")
axes[1].set_ylabel("Precision@K")
axes[1].set_ylim(0, 1.0 / K_RERANK + 0.05)
for i, v in enumerate(stage2[f"Precision@{K_RERANK}"]):
    axes[1].text(i, v + 0.001, f"{v:.4f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("retrieval_evaluation.png", dpi=150)
plt.show()
print("\n=== FULL RESULTS ===")
print("\nStage 1 – Recall@K (base retrievers):")
print(stage1.to_string(index=False))
print(f"\nStage 2 – Precision@K (after CrossEncoder reranking):")
print(stage2.to_string(index=False))